## Imports

In [34]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from math import radians, cos, sin, asin, sqrt

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Load Datasets

In [3]:
# Base dir path
BASE_DIR = Path('/content/drive/MyDrive/CVR College/Mini Project/data')

In [4]:
jalon_df = pd.read_csv(BASE_DIR / 'TRAIN_JALON.csv', encoding='latin')
lot_df = pd.read_csv(BASE_DIR / 'TRAIN_LOT.csv', encoding='latin')
gare_df = pd.read_csv(BASE_DIR / 'STATION.csv', encoding='latin')
wagon_df = pd.read_csv(BASE_DIR / 'TRAIN_WAGON.csv', encoding='latin')

/tmp/ipykernel_2207/370571192.py:1: DtypeWarning: Columns (19,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  jalon_df = pd.read_csv(BASE_DIR / 'TRAIN_JALON.csv', encoding='latin')
/tmp/ipykernel_2207/370571192.py:4: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  wagon_df = pd.read_csv(BASE_DIR / 'TRAIN_WAGON.csv', encoding='latin')


## Inspection

In [5]:
# Shape
print(f'Jalon: {jalon_df.shape}')
print(f'Lot: {lot_df.shape}')
print(f'Wagon: {wagon_df.shape}')
print(f'Gare: {gare_df.shape}')

Jalon: (72531, 27)
Lot: (14692, 45)
Wagon: (226998, 10)
Gare: (7174, 19)


In [6]:
# Duplicates in each df
print('Train Jalon: ', jalon_df.duplicated().sum())
print('Train Lot: ', lot_df.duplicated().sum())
print('Train Wagon: ', wagon_df.duplicated().sum())
print('Gare: ', gare_df['IDGARE'].duplicated().sum())

Train Jalon:  0
Train Lot:  0
Train Wagon:  0
Gare:  112


In [7]:
gare_df = gare_df[['IDGARE', 'Latitude', 'Longitude']]
print(gare_df.shape)
print(gare_df.columns)
print(gare_df['IDGARE'].duplicated().sum())

(7174, 3)
Index(['IDGARE', 'Latitude', 'Longitude'], dtype='object')
112


In [8]:
# Drop duplicates
print('Shape: ', gare_df.shape)
# Retain the first ID
gare_df = gare_df.drop_duplicates(subset=['IDGARE'], keep='first')
print(gare_df['IDGARE'].duplicated().sum())
print('Shape: ', gare_df.shape)

Shape:  (7174, 3)
0
Shape:  (7062, 3)


## Filter Datasets

In [9]:
jalon_df = jalon_df[[
    'IDTRAIN_JALON', 'IDTRAIN', 'IDGARE',
    'Jalon_num', 'Jalon_passage',
    'DHO_Arrivee', 'DHO_Depart',
    'DHT_Arrivee', 'DHT_Depart', 'DHR_Arrivee', 'DHR_Depart',
    'H_Arrivee_ecart', 'H_Depart_ecart',
    'H_Arrivee_ecart1', 'H_Depart_ecart1',
    'H_Arrivee_ecart2', 'H_Depart_ecart2',
    'Distance_origine',
]]

print(jalon_df.shape)
print(jalon_df.duplicated().sum())

(72531, 18)
0


In [10]:
lot_df = lot_df[[
    'IDTRAIN_LOT', 'IDTRAIN',
    'Distance',
    'Lot_num',
    'Nbre_TEU',
    'Poids_total',
    'Longueur_totale',
]]

# Drop trains with multiple lots
lot_df = lot_df[lot_df.Lot_num == 1]

print(lot_df.shape)
print(lot_df.duplicated().sum())

(14215, 7)
0


In [11]:
wagon_df = wagon_df[[
    'IDTRAIN_WAGON', 'IDTRAIN_LOT',
]]

# Extract number of lots from count of IDTRAIN_WAGON assigned to each lot
wagon_df = pd.DataFrame(wagon_df.groupby('IDTRAIN_LOT')['IDTRAIN_WAGON'].count()).reset_index()
# Rename columns
wagon_df.rename(columns={'IDTRAIN_WAGON': 'wagons_count'}, inplace=True)

print(wagon_df.shape)
print(wagon_df.columns)
print(wagon_df.duplicated().sum())

(11775, 2)
Index(['IDTRAIN_LOT', 'wagons_count'], dtype='object')
0


## Merge

In [12]:
print(jalon_df.shape)
# Merge jalon, lot
df = pd.merge(jalon_df, lot_df, on='IDTRAIN', how='left')
print(df.shape)

(72531, 18)
(72531, 24)


In [13]:
print(df.shape)
# Merge df, wagon
df = pd.merge(df, wagon_df, on='IDTRAIN_LOT', how='left')
print(df.shape)

(72531, 24)
(72531, 25)


In [14]:
print(df.shape)
# Merge df, gare
df = pd.merge(df, gare_df, on='IDGARE', how='left')
print(df.shape)

(72531, 25)
(72531, 27)


## Inspection

In [15]:
df.IDTRAIN.nunique()

14215

In [16]:
df.columns

Index(['IDTRAIN_JALON', 'IDTRAIN', 'IDGARE', 'Jalon_num', 'Jalon_passage',
       'DHO_Arrivee', 'DHO_Depart', 'DHT_Arrivee', 'DHT_Depart', 'DHR_Arrivee',
       'DHR_Depart', 'H_Arrivee_ecart', 'H_Depart_ecart', 'H_Arrivee_ecart1',
       'H_Depart_ecart1', 'H_Arrivee_ecart2', 'H_Depart_ecart2',
       'Distance_origine', 'IDTRAIN_LOT', 'Distance', 'Lot_num', 'Nbre_TEU',
       'Poids_total', 'Longueur_totale', 'wagons_count', 'Latitude',
       'Longitude'],
      dtype='object')

In [17]:
df.sort_values(by=['IDTRAIN', 'Jalon_num'], inplace=True)

In [18]:
# filter stations where jalon_passage == 0
df = df[df.Jalon_passage == 0]

In [26]:
mappings = {
    'y — arrival_delay_time': 'H_Arrivee_ecart (Next jalon)',
    'x1 — teu_count': 'Nbre_TEU',
    'x2 — train_length [m]': 'Longueur_totale',
    'x3 — total_distance_trip [km]': 'Distance',
    'x4 — departure_delay_time [min]': 'H_Depart_ecart (previous jalon)',
    'x5 — distance_between_stations [km]': '',
    'x6 — weight_per_length [t/m]': 'Poids_total / Longueur_totale',
    'x7 — weight_per_wagon [t/wagon]': 'Poids_total / wagons_count',
}

In [20]:
df.H_Depart_ecart.isna().sum()

np.int64(0)

In [21]:
df.wagons_count.isna().sum()

np.int64(10683)

In [22]:
df.head(10)

,IDTRAIN_JALON,IDTRAIN,IDGARE,Jalon_num,Jalon_passage,DHO_Arrivee,DHO_Depart,DHT_Arrivee,DHT_Depart,DHR_Arrivee,...,Distance_origine,IDTRAIN_LOT,Distance,Lot_num,Nbre_TEU,Poids_total,Longueur_totale,wagons_count,Latitude,Longitude
0,3507,255,80248,1,0,NaN,NaN,NaN,2019-03-04 01:50:00.000,NaN,...,NaN,255,99,1,0.0,353600.0,390.0,NaN,49.493889,6.108889
1,3508,255,84831,9999,0,NaN,NaN,2019-03-04 06:27:00.000,NaN,NaN,...,NaN,255,99,1,0.0,353600.0,390.0,NaN,48.736397,6.167817
8,3515,256,80248,1,0,NaN,NaN,NaN,2019-03-11 04:40:00.000,NaN,...,NaN,256,99,1,NaN,NaN,NaN,NaN,49.493889,6.108889
9,3516,256,84831,9999,0,NaN,NaN,2019-03-11 06:27:00.000,NaN,NaN,...,NaN,256,99,1,NaN,NaN,NaN,NaN,48.736397,6.167817
16,3531,258,80248,1,0,NaN,NaN,NaN,2019-03-25 04:40:00.000,NaN,...,NaN,258,99,1,NaN,NaN,NaN,NaN,49.493889,6.108889
17,3532,258,84831,9999,0,NaN,NaN,2019-03-25 06:27:00.000,NaN,NaN,...,NaN,258,99,1,NaN,NaN,NaN,NaN,48.736397,6.167817
24,3539,259,80248,1,0,NaN,NaN,NaN,2019-02-27 07:40:00.000,NaN,...,NaN,259,99,1,NaN,NaN,NaN,NaN,49.493889,6.108889
25,3540,259,84831,9999,0,NaN,NaN,2019-02-27 09:34:00.000,NaN,NaN,...,NaN,259,99,1,NaN,NaN,NaN,NaN,48.736397,6.167817
32,3547,260,80248,1,0,NaN,NaN,NaN,2019-03-06 07:40:00.000,NaN,...,NaN,260,99,1,NaN,NaN,NaN,NaN,49.493889,6.108889
33,3548,260,84831,9999,0,NaN,NaN,2019-03-06 09:34:00.000,NaN,NaN,...,NaN,260,99,1,NaN,NaN,NaN,NaN,48.736397,6.167817


In [28]:
temp_df = df[['IDTRAIN_JALON', 'IDTRAIN', 'Jalon_num', 'H_Arrivee_ecart', 'Nbre_TEU', 'Longueur_totale', 'Distance', 'H_Depart_ecart', 'Poids_total', 'wagons_count', 'Latitude', 'Longitude']]
temp_df.head()

,IDTRAIN_JALON,IDTRAIN,Jalon_num,H_Arrivee_ecart,Nbre_TEU,Longueur_totale,Distance,H_Depart_ecart,Poids_total,wagons_count,Latitude,Longitude
0,3507,255,1,0,0.0,390.0,99,60,353600.0,NaN,49.493889,6.108889
1,3508,255,9999,60,0.0,390.0,99,0,353600.0,NaN,48.736397,6.167817
8,3515,256,1,0,NaN,NaN,99,60,NaN,NaN,49.493889,6.108889
9,3516,256,9999,60,NaN,NaN,99,0,NaN,NaN,48.736397,6.167817
16,3531,258,1,0,NaN,NaN,99,60,NaN,NaN,49.493889,6.108889


In [35]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """
    Calculate great-circle distance between two points on Earth (in km).

    Parameters
    ----------
    lat1, lon1 : float
        Latitude and longitude of point 1 (degrees)
    lat2, lon2 : float
        Latitude and longitude of point 2 (degrees)

    Returns
    -------
    float
        Distance in kilometers
    """
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])

    # Haversine formula
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * asin(sqrt(a))
    r = 6371  # Earth's radius in km

    return c * r

def compute_consecutive_distances(group):
    """
    Compute distance between consecutive jalons for a single train.

    Parameters
    ----------
    group : pd.DataFrame
        Subset of data for a single train, sorted by jalon_num

    Returns
    -------
    pd.Series
        Distance (km) between current and previous jalon
    """
    distances = []

    for idx in range(len(group)):
        if idx == 0:
            # First jalon (departure) → distance = 0
            distances.append(0.0)
        else:
            lat1, lon1 = group.iloc[idx - 1][['Latitude', 'Longitude']]
            lat2, lon2 = group.iloc[idx][['Latitude', 'Longitude']]

            # Check for null coordinates
            if pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
                distances.append(np.nan)
            else:
                dist = haversine_distance(lat1, lon1, lat2, lon2)
                distances.append(dist)

    return pd.Series(distances, index=group.index)

def compute_cumulative_distance(group):
    """
    Compute cumulative distance from departure station.

    Parameters
    ----------
    group : pd.DataFrame
        Subset of data for a single train, sorted by jalon_num

    Returns
    -------
    pd.Series
        Cumulative distance (km) from departure
    """
    distances = []
    cumulative = 0.0

    for idx in range(len(group)):
        lat1, lon1 = group.iloc[idx - 1][['Latitude', 'Longitude']] if idx > 0 else (None, None)
        lat2, lon2 = group.iloc[idx][['Latitude', 'Longitude']]

        if idx == 0:
            cumulative = 0.0
        elif pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2):
            cumulative = np.nan
        else:
            dist = haversine_distance(lat1, lon1, lat2, lon2)
            cumulative += dist

        distances.append(cumulative)

    return pd.Series(distances, index=group.index)

In [36]:
print("Consecutive Distance (between adjacent jalons)")
print("="*70)

df_option1 = df.copy()

# Sort by train ID and jalon number
df_option1 = df_option1.sort_values(['IDTRAIN', 'Jalon_num']).reset_index(drop=True)

# Compute consecutive distances per train
df_option1['distance_to_previous_jalon_km'] = (
    df_option1.groupby('IDTRAIN', group_keys=False)
    .apply(compute_consecutive_distances)
)

print("\nSample output (first 20 rows, consecutive distance):")
print(df_option1[['IDTRAIN', 'Jalon_num', 'Latitude', 'Longitude',
                  'distance_to_previous_jalon_km']].head(20))

print("\nStatistics (consecutive distance):")
print(df_option1['distance_to_previous_jalon_km'].describe())

Consecutive Distance (between adjacent jalons)

Sample output (first 20 rows, consecutive distance):
    IDTRAIN  Jalon_num   Latitude  Longitude  distance_to_previous_jalon_km
0       255          1  49.493889   6.108889                       0.000000
1       255       9999  48.736397   6.167817                      84.338382
2       256          1  49.493889   6.108889                       0.000000
3       256       9999  48.736397   6.167817                      84.338382
4       258          1  49.493889   6.108889                       0.000000
5       258       9999  48.736397   6.167817                      84.338382
6       259          1  49.493889   6.108889                       0.000000
7       259       9999  48.736397   6.167817                      84.338382
8       260          1  49.493889   6.108889                       0.000000
9       260       9999  48.736397   6.167817                      84.338382
10      261          1  49.493889   6.108889                   

/tmp/ipykernel_2207/3640914017.py:12: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_consecutive_distances)


In [30]:
temp_df[['IDTRAIN_JALON', 'IDTRAIN', 'H_Depart_ecart']]

,IDTRAIN_JALON,IDTRAIN,H_Depart_ecart
0,3507,255,60
1,3508,255,0
8,3515,256,60
9,3516,256,0
16,3531,258,60
...,...,...,...
72526,120915,22417,60
72529,120918,22417,60
72530,120919,22417,60
72527,120916,22417,60


In [38]:
df_option1.columns

Index(['IDTRAIN_JALON', 'IDTRAIN', 'IDGARE', 'Jalon_num', 'Jalon_passage',
       'DHO_Arrivee', 'DHO_Depart', 'DHT_Arrivee', 'DHT_Depart', 'DHR_Arrivee',
       'DHR_Depart', 'H_Arrivee_ecart', 'H_Depart_ecart', 'H_Arrivee_ecart1',
       'H_Depart_ecart1', 'H_Arrivee_ecart2', 'H_Depart_ecart2',
       'Distance_origine', 'IDTRAIN_LOT', 'Distance', 'Lot_num', 'Nbre_TEU',
       'Poids_total', 'Longueur_totale', 'wagons_count', 'Latitude',
       'Longitude', 'distance_to_previous_jalon_km'],
      dtype='object')

In [39]:
def extract_departure_delay_previous_jalon(df):
    """
    Extract H_Depart_ecart from the previous jalon for each row.

    Parameters
    ----------
    df : pd.DataFrame
        Extracted dataset with columns: IDTRAIN, Jalon_num, H_Depart_ecart

    Returns
    -------
    pd.DataFrame
        Original dataframe with new column 'departure_delay_time_previous_jalon_min'
    """
    df_processed = df.copy()

    # Sort by train ID and jalon number to ensure proper ordering
    df_processed = df_processed.sort_values(['IDTRAIN', 'Jalon_num']).reset_index(drop=True)

    # Shift H_Depart_ecart within each train group
    # groupby().shift() handles the grouping automatically
    df_processed['departure_delay_time_previous_jalon_min'] = (
        df_processed.groupby('IDTRAIN')['H_Depart_ecart'].shift(1)
    )

    return df_processed

In [40]:
# Apply feature extraction
df_with_feature = extract_departure_delay_previous_jalon(df_option1)

In [41]:
df_with_feature.columns

Index(['IDTRAIN_JALON', 'IDTRAIN', 'IDGARE', 'Jalon_num', 'Jalon_passage',
       'DHO_Arrivee', 'DHO_Depart', 'DHT_Arrivee', 'DHT_Depart', 'DHR_Arrivee',
       'DHR_Depart', 'H_Arrivee_ecart', 'H_Depart_ecart', 'H_Arrivee_ecart1',
       'H_Depart_ecart1', 'H_Arrivee_ecart2', 'H_Depart_ecart2',
       'Distance_origine', 'IDTRAIN_LOT', 'Distance', 'Lot_num', 'Nbre_TEU',
       'Poids_total', 'Longueur_totale', 'wagons_count', 'Latitude',
       'Longitude', 'distance_to_previous_jalon_km',
       'departure_delay_time_previous_jalon_min'],
      dtype='object')

In [43]:
df_with_feature['weight_per_length [t/m]'] = df_with_feature['Poids_total']/df_with_feature['Longueur_totale']
df_with_feature['weight_per_wagon [t/wagon]'] = df_with_feature['Poids_total']/df_with_feature['wagons_count']

In [47]:
save_features = ['IDTRAIN_JALON', 'IDTRAIN', 'H_Arrivee_ecart', 'Nbre_TEU', 'Longueur_totale',
                 'Distance', 'departure_delay_time_previous_jalon_min', 'distance_to_previous_jalon_km',
                 'weight_per_length [t/m]', 'weight_per_wagon [t/wagon]']
df_save = df_with_feature[save_features]
df_save.columns

Index(['IDTRAIN_JALON', 'IDTRAIN', 'H_Arrivee_ecart', 'Nbre_TEU',
       'Longueur_totale', 'Distance',
       'departure_delay_time_previous_jalon_min',
       'distance_to_previous_jalon_km', 'weight_per_length [t/m]',
       'weight_per_wagon [t/wagon]'],
      dtype='object')

In [44]:
mappings

{'y — arrival_delay_time': 'H_Arrivee_ecart (Next jalon)',
 'x1 — teu_count': 'Nbre_TEU',
 'x2 — train_length [m]': 'Longueur_totale',
 'x3 — total_distance_trip [km]': 'Distance',
 'x4 — departure_delay_time [min]': 'H_Depart_ecart (previous jalon)',
 'x5 — distance_between_stations [km]': '',
 'x6 — weight_per_length [t/m]': 'Poids_total / Longueur_totale',
 'x7 — weight_per_wagon [t/wagon]': 'Poids_total / wagons_count'}

In [48]:
# Rename features
rename_mappings = {
    'H_Arrivee_ecart': 'arrival_delay_time',
    'Nbre_TEU': 'teu_count',
    'Longueur_totale': 'train_length [m]',
    'Distance': 'total_distance_trip [km]',
    'departure_delay_time_previous_jalon_min': 'departure_delay_time [min]',
    'distance_to_previous_jalon_km': 'distance_between_stations [km]',
    'weight_per_length [t/m]': 'weight_per_length [t/m]',
    'weight_per_wagon [t/wagon]': 'weight_per_wagon [t/wagon]',
}
df_save.rename(columns=rename_mappings, inplace=True)

/tmp/ipykernel_2207/2106455222.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_save.rename(columns=rename_mappings, inplace=True)


In [50]:
df_save.to_csv('extracted_datasetV3.csv')